# No cloning and dead qubits — the error-message tour

**The punchline.** Every exception `qsim` raises is a theorem wearing a stack trace. None
of them are the library being fussy: each one marks a place where the operation you asked
for does not exist in physics, and the message says which physical fact stops it.

This exhibit deliberately triggers four of them and reads the messages, then shows the
experiment behind the least obvious one.

Background: **[01 — States and gates](../01-states-and-gates.ipynb)** for what a qubit
handle is, and **[04 — Programs made of gates](../04-combinators.ipynb)** §4 for ancillas
and uncomputation.

In [ ]:
import copy
from collections.abc import Callable

import matplotlib.pyplot as plt
import numpy as np

from qsim import Circuit, DeadQubitError, DirtyAncillaError, NoCloningError, QsimError
from qsim.gates import CNOT, H, X


def expect(error_type: type[Exception], body: Callable[[], object]) -> Exception:
    """Run `body`, require it to raise `error_type`, print the message, return it.

    Every demonstration below goes through this, so if one of these impossible
    operations ever quietly starts working, the notebook fails instead of printing
    nothing and looking fine.
    """
    try:
        body()
    except error_type as err:
        print(f"--- {error_type.__name__} " + "-" * (58 - len(error_type.__name__)))
        print(err)
        return err
    raise AssertionError(f"expected {error_type.__name__}; nothing was raised")

## 1. `NoCloningError` — a qubit cannot control an operation on itself

`CNOT(a, b)` flips `b` where `a` is $\lvert 1\rangle$. So what should `CNOT(a, a)` do?

To answer, the machine would have to *read* `a` in order to decide what to do to `a` —
and having read it, it would hold a copy of the answer. The **no-cloning theorem** says
that is impossible for an unknown state, and the reason is one line of linear algebra:
copying is not a linear map. If some unitary sent $\lvert\psi\rangle\lvert 0\rangle$ to
$\lvert\psi\rangle\lvert\psi\rangle$ for every $\psi$, then by linearity it would send
$(\lvert 0\rangle + \lvert 1\rangle)\lvert 0\rangle$ to
$\lvert 00\rangle + \lvert 11\rangle$ — which is not
$(\lvert 0\rangle + \lvert 1\rangle)(\lvert 0\rangle + \lvert 1\rangle)$.
Quantum evolution *is* linear, so no such unitary exists.

In [ ]:
qc = Circuit(name="tour", seed=0)
a, b = qc.alloc_many(2)

err_self_cnot = expect(NoCloningError, lambda: CNOT(a, a))

## 2. `NoCloningError` — and Python's own copy protocol

A `Qubit` in qsim is a *handle*: a name for one axis of the circuit's state tensor, not a
value that carries a state around with it. Copying the handle would be harmless
bookkeeping — but it would read like a duplicate qubit, and the whole design of this
library is that nothing should read like something physics forbids. So `Qubit` refuses
`copy.copy` and `copy.deepcopy` too, and the message says what to do instead.

In [ ]:
err_copy = expect(NoCloningError, lambda: copy.copy(a))
print()
err_deepcopy = expect(NoCloningError, lambda: copy.deepcopy(a))

## 3. `NoCloningError` again — a control that is also a target

The same theorem shows up one level higher. `with qc.control(c):` runs a whole block only
where `c` is $\lvert 1\rangle$ — and if `c` is in superposition, the result is a
superposition of the block having run and not having run. Put an operation on `c` *inside*
that block and you are back to "read it to decide what to do to it".

The scope catches this when it closes, because that is when it sees the whole block. Note
that the state is untouched: a scope that raises discards everything it recorded rather
than running half a transformed block.

In [ ]:
def control_myself() -> None:
    with qc.control(a):
        X(b)
        H(a)      # <- a controls this block and is acted on inside it


err_self_control = expect(NoCloningError, control_myself)
print()
print("ops actually executed by that block:", len(qc.history))

## 4. `DeadQubitError` — a handle that names nothing

`with qc.ancilla(n) as scratch:` borrows scratch qubits and gives them back at the end of
the block. "Gives back" is literal: the axes are removed from the state tensor and the
handles are retired. A retired handle does not name a qubit any more.

The library could have let the handle silently point at whichever qubit shifted into that
axis position. That is the worst kind of bug — the program keeps running and computes
something else — so it refuses instead. (This is also why handles store a stable *id* and
ask the circuit where that id currently lives, rather than remembering an axis number.)

In [ ]:
qc2 = Circuit(name="ancilla-tour", seed=0)
q2 = qc2.alloc("q")

with qc2.ancilla(1) as scratch:
    s = scratch[0]
    X(s)
    X(s)          # clean again: X twice is the identity, so the scope is satisfied

print("inside the scope the circuit had 2 qubits; now it has", qc2.n_qubits)
print("the handle still exists as a Python object:", repr(s))
print()
err_dead = expect(DeadQubitError, lambda: X(s))

## 5. `DirtyAncillaError` — the one that is really about interference

The other three errors are about identity and lifetimes. This one is about physics you
can measure.

An ancilla scope checks, *numerically*, that the scratch qubits are back in
$\lvert 0\dots0\rangle$ **and unentangled** before releasing them. Leave a `CNOT` behind
and the scratch qubit holds a record of which branch the computation took — and the scope
refuses.

In [ ]:
qc3 = Circuit(name="dirty", seed=0)
q3 = qc3.alloc("q")
H(q3)             # q3 is now in a superposition of two branches


def leave_a_record() -> None:
    with qc3.ancilla(1) as scratch:
        CNOT(q3, scratch[0])     # the scratch qubit learns which branch q3 is in
        # ... and nothing here puts it back


err_dirty = expect(DirtyAncillaError, leave_a_record)

That message makes a claim — "a branch that has been recorded can no longer interfere
with the others" — and the claim is testable. Here is the test.

The circuit below is a two-slit experiment on one qubit: $H$ opens two paths, $H$ closes
them, and with nothing in between the two paths interfere constructively and the answer
is $\lvert 0\rangle$ with certainty. Between the Hadamards we do one of three things:
nothing, write a record into a second qubit, or write the record and then erase it again.

In [ ]:
def final_p0(record: str) -> float:
    """P(q = 0) after H - (optional record) - H. `record` is 'none', 'kept' or 'erased'."""
    qc = Circuit(name=f"two-slit-{record}", seed=0)
    q = qc.alloc("q")
    scratch = qc.alloc("scratch")

    H(q)                             # open two paths
    if record in ("kept", "erased"):
        CNOT(q, scratch)             # scratch learns which path
    if record == "erased":
        CNOT(q, scratch)             # ... and forgets it again: CNOT is its own inverse
    H(q)                             # recombine

    # The scratch qubit is not measured, only ignored. Ignoring it *is* the partial
    # trace, and reduced_density_matrix does exactly that; entry [0, 0] is P(q = 0).
    return float(np.real(qc.inspect.reduced_density_matrix([q])[0, 0]))


labels = ["no record", "record kept\n(dirty ancilla)", "record erased\n(uncomputed)"]
values = [final_p0("none"), final_p0("kept"), final_p0("erased")]

fig, ax = plt.subplots(figsize=(6.6, 3.4))
bars = ax.bar(labels, values, color=["teal", "crimson", "teal"], width=0.55)
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.axhline(0.5, color="gray", lw=0.8, ls="--")
ax.set_ylabel("P(0) after recombining")
ax.set_ylim(0.0, 1.18)
ax.set_title("what a leftover record costs: the interference, exactly")
fig.tight_layout()

for name, value in zip(["no record", "record kept", "record erased"], values, strict=True):
    print(f"{name:>16}:  P(0) = {value:.6f}")

Interference, gone and back. With no record the two paths recombine into certainty. With
the record left in place, $P(0)$ is a fair coin — *exactly* what you would get if the
qubit had been measured, even though nobody measured anything and the scratch qubit is
sitting there untouched. Undo the record and certainty returns.

So `DirtyAncillaError` is not a tidiness rule. A quantum algorithm's advantage comes from
arranging for wrong answers to cancel when the paths meet; a leftover record stops them
from meeting at all. The exception is the library refusing to let you silently lose the
thing you came for.

The middle bar is the same experiment as
**[decoherence_dial](decoherence_dial.ipynb)**, where the record is written by degrees
instead of all at once, and **[quantum_eraser](quantum_eraser.ipynb)**, which is the third
bar taken seriously.

## 6. One more, for the shape of it: `QsimError` inside a scope

Measurement is the only non-unitary operation in the library, so it is the only one that
cannot be inverted or conditioned on a superposed control. Both combinator scopes refuse
it, and the message explains why in terms of what the scopes *do*.

In [ ]:
qc4 = Circuit(name="scoped", seed=0)
q4, r4 = qc4.alloc_many(2)


def measure_inside_a_scope() -> None:
    with qc4.adjoint():
        H(q4)
        qc4.measure(q4)


err_scope = expect(QsimError, measure_inside_a_scope)

## What the four messages have in common

| Exception | The operation you asked for | The theorem in the way |
|---|---|---|
| `NoCloningError` | duplicate an unknown state, or let a qubit control itself | copying is not linear; quantum evolution is |
| `DeadQubitError` | use a handle whose axis is gone | a handle names a tensor factor, and that factor no longer exists |
| `DirtyAncillaError` | discard an entangled scratch qubit | discarding is a partial trace; a record destroys interference |
| `QsimError` (scopes) | invert or control a measurement | measurement discards branches, and nothing brings them back |

None of these could be fixed by a better implementation. That is the point of reading
them.

## Where to go next

- **[04 — Programs made of gates](../04-combinators.ipynb)** §4 and §5: uncomputation,
  and Bennett's trick for keeping an answer while giving the scratch back clean.
- **[quantum_eraser](quantum_eraser.ipynb)**: the third bar of the plot above, run at
  full strength and then undone.

## Assertions

The claims above, re-checked numerically.

In [ ]:
# 1. Each exception really is of the type claimed, and each message teaches the theorem.
assert isinstance(err_self_cnot, NoCloningError)
assert "no-cloning theorem" in str(err_self_cnot)
assert isinstance(err_copy, NoCloningError) and isinstance(err_deepcopy, NoCloningError)
assert "no-cloning theorem" in str(err_copy)
assert isinstance(err_self_control, NoCloningError)
assert "control an operation on itself" in str(err_self_control)
assert isinstance(err_dead, DeadQubitError)
assert "has been released and no longer refers to a qubit" in str(err_dead)
assert isinstance(err_dirty, DirtyAncillaError)
assert "uncomputed back to |0>" in str(err_dirty)
assert isinstance(err_scope, QsimError)
assert "irreversible" in str(err_scope)

# 2. A scope that raises leaves the state untouched: nothing from its body ran.
assert qc.history == []
assert qc.n_qubits == 2

# 3. The released ancilla's axis is gone from the state tensor.
assert qc2.n_qubits == 1
assert qc2.inspect.state_tensor().shape == (2,)

# 4. The physics behind DirtyAncillaError: a kept record costs exactly the interference.
assert np.isclose(final_p0("none"), 1.0)
assert np.isclose(final_p0("kept"), 0.5)
assert np.isclose(final_p0("erased"), 1.0)

print("all assertions passed")